# 05C High-Throughput COF Screening: From a Model to Candidate Materials

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/en/05C_high_throughput_screening.ipynb)

05B focused on **real CIF → descriptor table**. This chapter asks what happens after a model is trained.

We borrow the research architecture of `jsdvos/SupportingInformation_CO2captureHTS_2024`, which separates benchmark, ideal screening, machine learning, mixture screening, and analysis. Its ML stage uses feature/result tables, train/test structure lists, feature reduction, model training, and SHAP analysis.

This notebook uses a small teaching dataset so it remains fully runnable; it does not reproduce the full large-scale study.

## 1. Research-scale screening logic

```text
COF library → descriptors → expensive reference calculations on a subset
            → features + targets → train/validation/test → ML surrogate
            → predict a larger library → rank candidates
            → interpret → return to CIF → validate top candidates
```

The central idea is not simply choosing a more complex model. It is using a limited number of expensive calculations to train a surrogate that can screen many more structures.

## 2. Connection to a public COF CO₂-screening repository

Useful concepts in `jsdvos/SupportingInformation_CO2captureHTS_2024`: `Step2_MachineLearning`, `features.csv`, `results.csv`, `structs_train.txt`, `structs_test.txt`, feature reduction, model training, SHAP analysis, and prediction over a much larger hypothetical-COF library.

A major lesson is software organization: data generation, modeling, screening, and analysis can be separate, but structure IDs must preserve provenance across all stages.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
rng=np.random.default_rng(42)


## 3. A reduced screening table

This chapter uses a teaching-only candidate table. The target is generated by a pedagogical formula and **must not be used for scientific conclusions**.

05B and 05C serve different purposes: 05B is real CIF → descriptors; 05C is reference subset → surrogate model → large-library screening.

In [ ]:
n=500
candidates=pd.DataFrame({
 'COF_ID':[f'candidate_{i:04d}' for i in range(n)],
 'PLD_A':rng.uniform(3.0,18.0,n),
 'LCD_A':rng.uniform(5.0,35.0,n),
 'ASA_m2_g':rng.uniform(200,4500,n),
 'void_fraction':rng.uniform(0.25,0.90,n),
 'density_g_cm3':rng.uniform(0.25,1.40,n),
 'N_fraction':rng.uniform(0.0,0.20,n),
 'O_fraction':rng.uniform(0.0,0.20,n)})
noise=rng.normal(0,0.35,n)
candidates['CO2_uptake_demo']=(0.0010*candidates['ASA_m2_g']+2.2*candidates['N_fraction']+1.2*candidates['O_fraction']+0.8*candidates['void_fraction']-0.045*np.abs(candidates['PLD_A']-7.0)+noise)
candidates.head()


## 4. Only a subset receives expensive reference calculations

Assume reference calculations are affordable for only 120 out of 500 candidates. The model learns from this subset and predicts the remaining structures.

In [ ]:
feature_cols=['PLD_A','LCD_A','ASA_m2_g','void_fraction','density_g_cm3','N_fraction','O_fraction']
reference=candidates.sample(120,random_state=42).copy(); unseen=candidates.drop(reference.index).copy()
X_train,X_test,y_train,y_test=train_test_split(reference[feature_cols],reference['CO2_uptake_demo'],test_size=0.25,random_state=42)
model=RandomForestRegressor(n_estimators=400,random_state=42,n_jobs=-1)
model.fit(X_train,y_train); test_pred=model.predict(X_test)
print('MAE =',mean_absolute_error(y_test,test_pred)); print('R2 =',r2_score(y_test,test_pred))


## 5. Screen the uncalculated candidate library

Only after validation should the surrogate be applied to candidates without reference targets.

In [ ]:
unseen['predicted_CO2_uptake_demo']=model.predict(unseen[feature_cols])
top20=unseen.sort_values('predicted_CO2_uptake_demo',ascending=False).head(20)
display(top20[['COF_ID','predicted_CO2_uptake_demo']+feature_cols])


## 6. Ranking is not the end: check applicability domain

A high prediction may be an extrapolation. This simple min–max check is only a first warning; research workflows can use distances, uncertainty estimates, ensembles, or disagreement metrics.

In [ ]:
train_min=reference[feature_cols].min(); train_max=reference[feature_cols].max()
outside=((top20[feature_cols]<train_min)|(top20[feature_cols]>train_max)).any(axis=1)
top20=top20.assign(outside_training_range=outside.values)
display(top20[['COF_ID','predicted_CO2_uptake_demo','outside_training_range']])


## 7. Interpretation

The published COF workflow uses SHAP. For a beginner baseline, start with random-forest feature importance and later replace it with SHAP or more rigorous interpretation tools.

In [ ]:
importance=pd.Series(model.feature_importances_,index=feature_cols).sort_values()
plt.figure(figsize=(6,4)); importance.plot(kind='barh'); plt.xlabel('Random-forest feature importance'); plt.show()


## 8. What a real project still needs

A research-grade workflow should use reproducible descriptors from CIF/pore-analysis software, targets at consistent thermodynamic/computational conditions, stable COF IDs, duplicate control, family/topology-aware splitting, leakage-free model selection, interpretation, applicability-domain checks, and high-accuracy revalidation of top candidates.

The ML output is a **candidate list for validation**, not proof that the model has discovered the best COF.

## 9. Repositories worth studying

- **CURATED-COFs**: experimental COF CIFs and structure-curation provenance.
- **CoRE-COF Database**: larger database versions and structure screening.
- **SupportingInformation_CO2captureHTS_2024**: CO₂ capture HTS + ML + SHAP workflow.
- **mofdscribe**: porous-material featurization, benchmarking, and splitting concepts that transfer well to COFs.

Do not copy scripts blindly. Extract the reusable research structure: **data generation → representation → validation → screening → interpretation → verification**.

## Exercises

1. Change the reference size to 40, 80, and 200; compare MAE and top-20 stability.
2. Remove pore descriptors and keep only composition-like features, then do the reverse.
3. Design a second-stage expensive-calculation queue for top candidates.
4. Explain why the same test set must not be repeatedly used for tuning and final reporting.
5. Design a real project directory containing `cifs/`, `features/`, `targets/`, `splits/`, `models/`, and `predictions/`.

### True Level-A completion criterion
You can explain where structures come from, how descriptors are generated, how targets are defined, how tables are aligned, how splits are designed, how a model is validated, how unseen COFs are screened, and why top structures still require physical validation.
